# Propensity Analysis – Multi‑Model Feature Accretion
This notebook loads *all* JSONL outputs and compares models on novelty, repetition, and breakdown signals.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

def find_outputs_dir() -> Path:
    # Walk up from current working dir to find experiments/outputs
    here = Path.cwd().resolve()
    for _ in range(6):
        candidate = here / 'experiments' / 'outputs'
        if candidate.exists():
            return candidate
        here = here.parent
    # Fallback to repo path if running from this notebook's folder
    candidate = Path('experiments/outputs')
    return candidate

OUTPUT_DIR = find_outputs_dir()
files = sorted(OUTPUT_DIR.glob('*.jsonl'))
print(f'Loading {len(files)} files from {OUTPUT_DIR}')

records = []
summaries = []
for path in files:
    model = path.stem.replace('local_llm__', '')
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        rec['model'] = model
        if rec.get('record_type') == 'prompt_response':
            records.append(rec)
        elif rec.get('record_type') == 'run_summary':
            summaries.append(rec)

df = pd.DataFrame(records)
df_summary = pd.DataFrame(summaries)
# Focus on feature_accretion runs if present
if 'mode' in df.columns:
    df = df[df['mode'] == 'feature_accretion']
df.head()


In [ ]:
df_summary

## Quick Metrics by Model

In [ ]:
metrics = df.groupby('model').agg(
    steps=('step','max'),
    avg_novelty=('novelty_score','mean'),
    avg_repetition=('repetition_rate','mean'),
    avg_similarity=('similarity_to_prev','mean'),
    hit_max_rate=('hit_max_gen_tokens','mean'),
    format_ok_rate=('format_ok','mean') if 'format_ok' in df else ('step','count')
)
metrics


## Novelty Over Time (per model)

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=df, x='step', y='novelty_score', hue='model')
plt.axhline(0.25, color='red', linestyle='--', label='novelty threshold')
plt.title('Novelty score by step')
plt.legend()
plt.show()

## Repetition and Similarity Trends

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.lineplot(data=df, x='step', y='repetition_rate', hue='model', ax=ax[0])
ax[0].set_title('Repetition rate')
sns.lineplot(data=df, x='step', y='similarity_to_prev', hue='model', ax=ax[1])
ax[1].set_title('Similarity to previous')
plt.tight_layout()
plt.show()

## Breakdown Score and Truncation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.lineplot(data=df, x='step', y='breakdown_score', hue='model', ax=ax[0])
ax[0].set_title('Breakdown score')
sns.lineplot(data=df, x='step', y='hit_max_gen_tokens', hue='model', ax=ax[1])
ax[1].set_title('Hit max gen tokens (rate)')
plt.tight_layout()
plt.show()

## Feature Ledger Growth

In [ ]:
if 'feature_ledger_count' in df:
    plt.figure(figsize=(10,4))
    sns.lineplot(data=df, x='step', y='feature_ledger_count', hue='model', marker='o')
    plt.title('Feature ledger count')
    plt.show()

## Memory Check – Language Recall

In [ ]:
if 'memory_match' in df:
    plt.figure(figsize=(10,4))
    sns.lineplot(data=df, x='step', y='memory_match', hue='model')
    plt.title('Memory Match Rate by Step')
    plt.xlabel('Step')
    plt.ylabel('Match (1=true, 0=false)')
    plt.show()

if 'memory_distance' in df:
    plt.figure(figsize=(10,4))
    sns.lineplot(data=df, x='step', y='memory_distance', hue='model')
    plt.title('Memory Edit Distance by Step')
    plt.xlabel('Step')
    plt.ylabel('Edit Distance (lower is better)')
    plt.show()


## Memory Contamination vs Drift

In [ ]:
if 'memory_contamination' in df:
    plt.figure(figsize=(10,4))
    sns.lineplot(data=df, x='step', y='memory_contamination', hue='model')
    plt.title('Memory Contamination Rate by Step')
    plt.xlabel('Step')
    plt.ylabel('Contamination (1=true)')
    plt.show()

if 'memory_probe_used' in df and 'memory_contamination' in df:
    probe = df[df['memory_probe_used'] == True]
    non_probe = df[df['memory_probe_used'] == False]
    summary = {
        'probe_contamination_rate': probe['memory_contamination'].mean(),
        'non_probe_contamination_rate': non_probe['memory_contamination'].mean(),
    }
    summary


## Post‑Processed Memory Accuracy (from text)

In [ ]:
import re

def extract_memory_line(text):
    patterns = [
        r'MemoryCheck\s*:\s*([A-Za-z#+ -]+)',
        r'Language\s*:\s*([A-Za-z#+ -]+)',
        r'Previous\s+language\s*(?:was|:)\s*([A-Za-z#+ -]+)',
        r'Previously\s+coding\s+in\s*([A-Za-z#+ -]+)',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            ans = m.group(1).strip()
            return re.sub(r'[^\w#+\- ]+', '', ans).strip()
    # fallback to first line
    first = text.strip().splitlines()[0] if text.strip() else ''
    m = re.search(r'Language\s*:\s*([A-Za-z#+ -]+)', first, re.IGNORECASE)
    if m:
        ans = m.group(1).strip()
        return re.sub(r'[^\w#+\- ]+', '', ans).strip()
    return None

if 'memory_expected' in df.columns:
    df['memory_answer_post'] = df['response'].apply(extract_memory_line)
    df['memory_correct_post'] = (df['memory_answer_post'].str.lower() == df['memory_expected'].fillna('').str.lower()).astype('float')
    # Only count where we actually had a probe
    if 'memory_probe_used' in df:
        df.loc[df['memory_probe_used'] != True, 'memory_correct_post'] = np.nan
    df[['memory_expected','memory_answer_post','memory_correct_post']].head()


In [ ]:
if 'memory_correct_post' in df:
    memory_rates = df.groupby('model')['memory_correct_post'].mean()
    memory_rates


## Summary of Findings (Current Runs)

| model | steps | avg_novelty | avg_repetition | avg_similarity | hit_max_rate | format_ok_rate | memory_probe_steps | memory_extraction_rate | memory_correct_rate_post |
|--- | --- | --- | --- | --- | --- | --- | --- | --- | ---|
| Meta-Llama-3.1-8B-Instruct-3bit | 11 | 0.339 | 0.141 | 0.235 | 0.077 | 0.790 | 41 | 0.000 |  |
| Mistral-7B-Instruct-v0.3 | 16 | 0.345 | 0.108 | 0.196 | 0.447 | 0.981 | 45 | 0.000 |  |
| Phi-3-mini-4k-instruct-4bit | 8 | 0.428 | 0.107 | 0.063 | 0.262 | 0.550 | 36 | 0.000 |  |
| Phi-4-mini-instruct-8bit | 19 | 0.363 | 0.044 | 0.141 | 0.037 | 0.848 | 50 | 0.000 |  |
| SmolLM-1.7B-Instruct-4bit | 6 | 0.438 | 0.202 | 0.242 | 0.258 | 0.134 | 22 | 0.000 |  |
| mistral_propensity_features | 8 | 0.343 | 0.114 | 0.161 | 0.333 | 1 | 6 | 0.000 |  |

**Notes:**
- Memory correctness is computed via post-processed extraction from responses.
- If `memory_correct_rate_post` is blank, the model did not provide a parseable memory answer.
